## Setup

In [54]:
import xarray as xr
from pathlib import Path
import random
import datetime
import numpy as np

# directory with source files 
data_dir = Path('/import/beegfs/CMIP6/arctic-cmip6/Arctic_Rivers_Data')

# load combined outputs
wt_ds = xr.open_dataset('/import/beegfs/CMIP6/arctic-cmip6/Arctic_Rivers_Data/combined_wt.nc')
q_ds = xr.open_dataset('/import/beegfs/CMIP6/arctic-cmip6/Arctic_Rivers_Data/combined_q.nc')

# load daily climatologies
### TBD

## QC combined values against source data

In [55]:
def compare_combined_file_with_source_data(variable, combined_ds):
    # for each model, and a random stream id,
    # check a random historical date (1990-2021) and a random future date (2034-2065)
    # make sure the values match the source files
    # source file naming convention is YYYY_model_<variable>.nc, with variable being "WT" or "Q"
    # to find the file names, we add "h" prefix to the model name for historical and "f" prefix for future
    # (not applicable to the "historical" model, which is historical period only)

    models = combined_ds.model.values
    stream_id = random.choice(combined_ds.stream_id.values)
    historical_date = np.datetime64(datetime.date(random.randint(1990, 2021), random.randint(1, 12), random.randint(1, 28)))
    future_date = np.datetime64(datetime.date(random.randint(2034, 2065), random.randint(1, 12), random.randint(1, 28)))

    # we can't have any dates past 10-01-2021 for historical, or past 10-01-2065 for future. If we do, just set these to 10-01 of that year
    if historical_date > np.datetime64('2021-10-01'):
        historical_date = np.datetime64('2021-10-01')
    if future_date > np.datetime64('2065-10-01'):
        future_date = np.datetime64('2065-10-01')

    # there is also a date shifting problem with the NCAR model ensemble members.
    # this only affects dates in the first 2 weeks of January and the last 2 weeks of December of any year
    # for these dates, we will just shift them to Jan 15 or Dec 15
    if historical_date < np.datetime64(f"{historical_date.astype('datetime64[Y]').item().year}-01-15"):
        historical_date = np.datetime64(f"{historical_date.astype('datetime64[Y]').item().year}-01-15")
    if historical_date > np.datetime64(f"{historical_date.astype('datetime64[Y]').item().year}-12-15"):
        historical_date = np.datetime64(f"{historical_date.astype('datetime64[Y]').item().year}-12-15")
    if future_date < np.datetime64(f"{future_date.astype('datetime64[Y]').item().year}-01-15"):
        future_date = np.datetime64(f"{future_date.astype('datetime64[Y]').item().year}-01-15")
    if future_date > np.datetime64(f"{future_date.astype('datetime64[Y]').item().year}-12-15"):
        future_date = np.datetime64(f"{future_date.astype('datetime64[Y]').item().year}-12-15")
        
        
    for model in models:
        source_file_hist = data_dir / f"{historical_date.astype('datetime64[Y]').item().year}_{'h' + model if model != 'historical' else model}_{variable}.nc"
        source_file_fut = data_dir / f"{future_date.astype('datetime64[Y]').item().year}_{'f' + model if model != 'historical' else model}_{variable}.nc"

        if variable == "Q":
            sel_dict = {"source_file_variable": "IRFroutedRunoff",
                "source_file_id": "seg"}

        elif variable == "WT":
            sel_dict = {"source_file_variable": "T_stream",
                "source_file_id": "hru",
                "additional_filter": {"no_seg": 1}}
            
        # load source files
        src_ds_hist = xr.open_dataset(source_file_hist)
        src_ds_fut = xr.open_dataset(source_file_fut)

        # standardize the time dimension to be all 0 hour to match combined dataset
        src_ds_hist['time'] = src_ds_hist['time'].dt.floor('D')
        src_ds_fut['time'] = src_ds_fut['time'].dt.floor('D')

        # all timestamps should be present in the combined datasets - if we get a KeyError: "not all values found in index 'time', print a warning and skip this timestep
        try:
            _ = combined_ds.sel(model=model, stream_id=stream_id).sel(time=historical_date)
        except KeyError as e:
            print(f"Warning: Combined dataset is missing data for model {model}, stream_id {stream_id}, historical date {historical_date}. Probably missing a timestep? Skipping this test case.")
            return None
        try: 
            _ = combined_ds.sel(model=model, stream_id=stream_id).sel(time=future_date)
        except KeyError as e:
            print(f"Warning: Combined dataset is missing data for model {model}, stream_id {stream_id}, future date {future_date}. Probably missing a timestep? Skipping this test case.")
            return None
        
        # get combined ds values
        combined_value_hist = combined_ds.sel(model=model, stream_id=stream_id).sel(time=historical_date)[sel_dict["source_file_variable"]].item()
        combined_value_fut = combined_ds.sel(model=model, stream_id=stream_id).sel(time=future_date)[sel_dict["source_file_variable"]].item()


        # all timestamps might not be present in the source datasets - if we get a KeyError: "not all values found in index 'time', print a warning and skip this timestep
        # get source ds values
        if sel_dict.get("additional_filter") is not None:
            try:
                source_value_hist = src_ds_hist[sel_dict["source_file_variable"]].sel({sel_dict["source_file_id"]: stream_id}).sel(time=historical_date).sel(sel_dict["additional_filter"]).item()
            except KeyError as e:
                print(f"Warning: The source dataset is missing data for model {model}, stream_id {stream_id}, historical date {historical_date}. Probably missing a timestep? Skipping this test case.")
                print(e)
                return None
            try:
                source_value_fut = src_ds_fut[sel_dict["source_file_variable"]].sel({sel_dict["source_file_id"]: stream_id}).sel(time=future_date).sel(sel_dict["additional_filter"]).item()
            except KeyError as e:
                print(f"Warning: The source dataset is missing data for model {model}, stream_id {stream_id}, future date {future_date}. Probably missing a timestep? Skipping this test case.")
                print(e)
                return None
        else:
            try:
                source_value_hist = src_ds_hist[sel_dict["source_file_variable"]].sel({sel_dict["source_file_id"]: stream_id}).sel(time=historical_date).item()
            except KeyError as e:
                print(f"Warning: The source dataset is missing data for model {model}, stream_id {stream_id}, historical date {historical_date}. Probably missing a timestep? Skipping this test case.")
                print(e)
                return None
            try:
                source_value_fut = src_ds_fut[sel_dict["source_file_variable"]].sel({sel_dict["source_file_id"]: stream_id}).sel(time=future_date).item()
            except KeyError as e:
                print(f"Warning: The source dataset is missing data for model {model}, stream_id {stream_id}, future date {future_date}. Probably missing a timestep? Skipping this test case.")
                print(e)
                return None
            

        # check that values are the same, and print any mismatches
        if not np.isclose(combined_value_hist, source_value_hist):
            print(f"Mismatch in historical value for model {model}, stream_id {stream_id}, date {historical_date}: combined {combined_value_hist}, source {source_value_hist}")
        if not np.isclose(combined_value_fut, source_value_fut):
            print(f"Mismatch in future value for model {model}, stream_id {stream_id}, date {future_date}: combined {combined_value_fut}, source {source_value_fut}")

        return None

In [56]:
# no output = all values match, passes QC
# run this function 10x to check different random values
for _ in range(100):
    compare_combined_file_with_source_data("Q", q_ds)

In [57]:
# no output = all values match, passes QC
# run this function 10x to check different random values
for _ in range(100):
    compare_combined_file_with_source_data("WT", wt_ds)